# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0307,  0.1655, -0.1489,  0.0050,  0.0262,  0.1945, -0.0207, -0.2739],
        [ 0.0517, -0.0081,  0.0967, -0.0207,  0.0304, -0.0229, -0.0839,  0.0966]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 1.3572, -1.4288],
        [-0.1959,  0.2329],
        [ 0.0895,  0.4649],
        [ 0.0515, -0.0054],
        [ 0.9717,  0.6499]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.2783, -0.1245],
        [ 0.1263,  0.0268],
        [ 0.1335,  0.2475],
        [-0.0643,  0.2470],
        [ 0.4548,  0.1173]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [7]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    def forward(self, x):
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.2009, -1.4683],
        [-0.7965, -0.0408],
        [-0.1406, -1.1855],
        [-0.4827, -0.5998],
        [-0.8389, -0.4776]])
Output inference_mode method:
 tensor([[-0.2009, -1.4683],
        [-0.7965, -0.0408],
        [-0.1406, -1.1855],
        [-0.4827, -0.5998],
        [-0.8389, -0.4776]])
Output no_grad context manager:
 tensor([[-0.2009, -1.4683],
        [-0.7965, -0.0408],
        [-0.1406, -1.1855],
        [-0.4827, -0.5998],
        [-0.8389, -0.4776]])
Output inference_mode context manager:
 tensor([[-0.2009, -1.4683],
        [-0.7965, -0.0408],
        [-0.1406, -1.1855],
        [-0.4827, -0.5998],
        [-0.8389, -0.4776]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [8]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0596, 0.0000, 0.0000, 2.8242],
        [0.0000, 0.0000, 0.0000, 0.3777]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0298, 0.0000, 0.0000, 1.4121],
        [0.0000, 0.0000, 1.1035, 0.1889]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [13]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/LSTM.png?raw=1)

In [14]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.tanh = nn.Tanh()
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

        self.hidden_size = hidden_size

    def forward(self, input, hidden=None, cell=None):
        batch_size = input.shape[0]
        seq_len = input.shape[1]

        if not hidden:
            hidden = torch.zeros(size=(batch_size, self.hidden_size), dtype=torch.float32)

        if not cell:
            cell = torch.zeros(size=(batch_size, self.hidden_size), dtype=torch.float32)

        output = []

        for t in range(seq_len):
            new_input = torch.cat((input[:, t, :], hidden), dim=-1)
            f_t = self.sigmoid(self.W_f(new_input))
            c_t = self.tanh(self.W_c(new_input))
            i_t = self.sigmoid(self.W_i(new_input))
            cell = f_t * cell + i_t * c_t

            o_t = self.sigmoid(self.W_o(new_input))
            hidden = o_t * self.tanh(cell)
            output.append(hidden)

        return torch.stack(output, dim=1), (hidden, cell)

In [15]:
input = torch.randn(size=(2, 4, 5))
model = LSTM(input_size=5, hidden_size=10)

output, (h, c) = model(input)
print(output.shape, h.shape, c.shape)

torch.Size([2, 4, 10]) torch.Size([2, 10]) torch.Size([2, 10])


### Inception

![inception](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/inception.png?raw=1)

In [16]:
class Inception(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = nn.ModuleList([nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=1)
                                   for i in range(4)])

        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=2)
        self.relu = nn.ReLU()


    def forward(self, input):
        out = []
        out.append(self.conv1[0](input))
        out.append(self.conv3(self.relu(self.conv1[1](input))))
        out.append(self.conv5(self.relu(self.conv1[2](input))))
        out.append(self.conv1[3](self.relu(self.max_pool(input))))

        return torch.cat(out, dim=1) # B x C x H x W

In [17]:
input = torch.randn(size=(2, 3, 5, 7))
model = Inception(in_channels=3)

output = model(input)
print(output.shape)

torch.Size([2, 288, 5, 7])


### SE

![se](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SqueezeAndExcite.png?raw=1)

In [18]:
class SE(nn.Module):
    def __init__(self, channels, hidden_dim):
        super().__init__()

        self.squeeze = nn.AdaptiveAvgPool2d(output_size=1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, channels),
            nn.Sigmoid()
        )

    def forward(self, input):
        out = self.squeeze(input).permute(0, 2, 3, 1)
        out = self.excitation(out).permute(0, 3, 1, 2) # B x C x 1 x 1
        return input * out


In [19]:
input = torch.randn(size=(2, 3, 5, 7))
model = SE(channels=3, hidden_dim=10)

output = model(input)
print(output.shape)

torch.Size([2, 3, 5, 7])


### Selective Kernel

![selective](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SelectiveKernel.png?raw=1)


In [20]:
class SK(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim):
        super().__init__()

        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding='same')
        self.relu = nn.ReLU()
        self.norm1 = nn.BatchNorm2d(out_channels)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.conv5 = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding='same')
        self.pooling = nn.AdaptiveAvgPool2d(output_size=1)
        self.squeeze = nn.Linear(out_channels, hidden_dim)
        self.A = nn.Linear(hidden_dim, out_channels)
        self.B = nn.Linear(hidden_dim, out_channels)


    def forward(self, input):
        U1 = self.relu(self.norm1(self.conv3(input)))
        U2 = self.relu(self.norm2(self.conv5(input)))
        out = self.pooling(U1 + U2) # batch_size x out_channels x 1 x 1
        out = self.squeeze(out.squeeze())

        A = self.relu(self.A(out))
        B = self.relu(self.B(out)) # batch_size x out_channels
        F = torch.softmax(torch.stack((A, B), dim=2), dim=-1) # batch_size x out_channels x 2

        U1 = U1 * F[:, :, 0][:, :, None, None]
        U2 = U2 * F[:, :, 1][:, :, None, None]

        return U1 + U2 # batch_size x out_channels x H x W



In [21]:
input = torch.randn(size=(2, 3, 5, 7))
model = SK(in_channels=3, out_channels=5, hidden_dim=10)

output = model(input)
print(output.shape)

torch.Size([2, 5, 5, 7])


### PatchMerger

![patchmerger](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/PatchMerger.png?raw=1)


In [22]:
class PatchMerger(nn.Module):
    def __init__(self, N, D, M):
        super().__init__()
        self.W = nn.Linear(D, M)
        self.norm = nn.LayerNorm(D)

    def forward(self, input):
        # input = Tensor[B, N, D]
        out = torch.softmax(self.W(input).transpose(-1, -2), dim=-1)
        return out @ input

input = torch.randn(size=(2, 200, 10))
model = PatchMerger(N=200, D=10, M=10)

output = model(input)
print(output.shape)

torch.Size([2, 10, 10])


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ResBlock.png?raw=1)

https://arxiv.org/pdf/1512.03385

In [23]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding="same"),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size, padding="same"),
            nn.BatchNorm2d(channels)
        )

    def forward(self, input):
        return input + self.block(input)


input = torch.randn(size=(10, 5, 20, 15))
model = ResidualBlock(channels=5, kernel_size=3)

output = model(input)
print(output.shape)

torch.Size([10, 5, 20, 15])


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/DepthWiseConv.png?raw=1)

https://arxiv.org/pdf/1610.02357

In [24]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, kernel_size, out_channels):
        super().__init__()
        self.dwconv = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size, groups=in_channels)
        self.pconv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, input):
        return self.pconv(self.dwconv(input))

input = torch.randn(size=(2, 3, 20, 20))
model = SeparableConv2d(in_channels=3, out_channels=5, kernel_size=3)

output = model(input)
print(output.shape)

torch.Size([2, 5, 18, 18])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [25]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_a = nn.Linear(hidden_dim, hidden_dim)
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)


    def forward(self, keys, queries):
        # keys = Tensor[B x L x d]
        # queries = Tensor[B x d]
        score = keys * self.W_a(queries).unsqueeze(1)
        attn = torch.softmax(score.sum(dim=-1), dim=-1)
        context = torch.sum(attn.unsqueeze(-1) * keys, dim=1)

        out = torch.tanh(self.W_v(context) + self.W_q(queries))
        return out


keys = torch.randn(size=(2, 3, 20))
queries = torch.randn(size=(2, 20))
model = VanillaAttention(20)

output = model(keys, queries)
print(output.shape)

torch.Size([2, 20])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [26]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Q = nn.Linear(d_model, d_model)
        self.K = nn.Linear(d_model, d_model)
        self.V = nn.Linear(d_model, d_model)

        self.d_model = d_model


    def forward(self, input, mask=None):
        keys = self.K(input)
        queries = self.Q(input)
        values = self.V(input)

        score = queries @ keys.transpose(-1, -2) / self.d_model ** (0.5)
        if mask is not None:
            score = score.masked_fill(~mask.to(torch.bool), float("-inf"))

        attn = torch.softmax(score, dim=-1) @ values
        return attn

input = torch.randn(size=(2, 10, 20))
model = ScaledDotProductAttention(d_model=20)
mask = torch.tril(torch.ones(10, 10))

output = model(input, mask)
print(output.shape)

torch.Size([2, 10, 20])


## Multihead Attention (0.1 балл)

![MultiheadAttention](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MultiheadAttention.webp?raw=1)

https://arxiv.org/abs/1706.03762


In [27]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.Q = nn.Linear(d_model, d_model)
        self.K = nn.Linear(d_model, d_model)
        self.V = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        self.d_model = d_model
        self.num_heads = num_heads

        assert d_model % num_heads == 0


    def forward(self, input, mask=None):
        B, N, _ = input.shape
        keys = self.K(input).view(B, N, self.num_heads, -1).transpose(1, 2)
        queries = self.Q(input).view(B, N, self.num_heads, -1).transpose(1, 2)
        values = self.V(input).view(B, N, self.num_heads, -1).transpose(1, 2)

        score = queries @ keys.transpose(-1, -2) / self.d_model ** (0.5)

        if mask is not None:
            score = score.masked_fill(~mask.to(torch.bool), float("-inf"))

        attn = torch.softmax(score, dim=-1) @ values
        out = self.out_proj(attn.transpose(1, 2).reshape(B, N, self.d_model))

        return out

input = torch.randn(size=(2, 10, 20))
model = MultiHeadAttention(d_model=20, num_heads=4)
mask = torch.tril(torch.ones(10, 10))

output = model(input, mask)
print(output.shape)

torch.Size([2, 10, 20])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/TransformerEncoder.png?raw=1)


https://arxiv.org/abs/1706.03762

In [28]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, dropout=0.5):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout)
        )


    def forward(self, input, mask=None):
        out = self.attention(self.norm1(input), mask=mask) + input
        out = self.mlp(self.norm2(out)) + out
        return out

input = torch.randn(size=(2, 10, 20))
model = TransformerEncoderLayer(d_model=20, num_heads=4, hidden_dim=80)
mask = torch.tril(torch.ones(10, 10))

output = model(input, mask)
print(output.shape)

torch.Size([2, 10, 20])


## MLP Mixer (0.1 балл)


![MLPMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MLPMixer.png?raw=1)


https://arxiv.org/abs/2105.01601

In [29]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches, channels, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)

        self.mlp1 = nn.Sequential(
            nn.Linear(num_patches, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_patches)
        )
        self.mlp2 = nn.Sequential(
            nn.Linear(channels, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, channels)
        )

    def forward(self, input):
        # input = Tensor[batch_size, num_patches, channels]
        out = self.mlp1((self.norm1(input).transpose(-1, -2)))
        out = input + out.transpose(-1, -2)
        out = self.mlp2(self.norm2(out)) + out
        return out


input = torch.randn(size=(2, 3, 20, 20))
input = nn.Conv2d(3, 3, 4, stride=4, groups=3)(input).flatten(start_dim=-2)
model = MLPMixerBlock(num_patches=25, channels=3, hidden_dim=10)

output = model(input.transpose(-2, -1))
print(output.shape)


torch.Size([2, 25, 3])


## ConvMixer (0.1 балл)

![ConvMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ConvMixer.png?raw=1)


https://arxiv.org/abs/2201.09792

In [30]:
class ConvMixer(nn.Module):

    def __init__(self, channels: tuple, kernel_size: tuple):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(channels[0], channels[1], kernel_size, groups=channels[1], padding="same"),
            nn.ReLU(),
            nn.BatchNorm2d(channels[1])
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(channels[1], channels[2], kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(channels[2])
        )


    def forward(self, input):
        input = input + self.block1(input)
        return self.block2(input)

input = torch.randn(size=(2, 3, 20, 20))
model = ConvMixer(channels=(3, 3, 3), kernel_size=5)

output = model(input)
print(output.shape)


torch.Size([2, 3, 20, 20])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Обе архитектуры с Multihead Attention объединяет то, что они выстраивают связи среди токенов и обеспечивают обмен информацией между ними, только делают это по-своему. Multihead Attention учитывает влияние всех токенов друг на друга с помощью матрицы
$$
softmax \bigg(\frac{QK^T}{\sqrt{d}}\bigg).
$$
Не обладая таким механизмом, MLPMixer предлагает использовать для этой цели матрицу обученных весов:

$$
X = X + (X^TW_{MLP_1})^T \\
X = X + XW_{MLP_2}.
$$

При этом первое преобразование действует независимо на каждый токен и отвечает за смешивание признаков, а второе, наоборот, объединяет информацию токенов.

ConvMixer применяет свёртки:

$$
X = X + \text{Conv}_{DW}(X) \\
X = X + \text{Conv}_{PW}(X).
$$

Первая свёртка обеспечивает обмен информацией между токенами, тогда как вторая смешивает признаки. Таким образом, все три метода учитывают зависимости векторов друг от друга (причём как пространственные (emb), так и позиционные (pos)), поэтому последние два подхода иногда могут работать не хуже. Если объединить их общей формулой, можно написать примерно так:
$$
X = X + W_{emb}(X) + W_{pos}(X).
$$

Attention учитывает влияние всех токенов друг на друга, обладая широким контекстом, что позволяет ему более глубоко выстраивать между ними связи. Но это объясняет его главный недостаток: высокая вычислительная сложность на длинных последовательностях. Архитектура MLPMixer проще. В ней нет softmax, меньше параметров и вычислений. Но она применяет статическую матрицу весов, одинаковую для всех входных последовательностей (в отличие от Attention). И хотя по-прежнему обмен информацией происходит между всеми токенами, у него больше ограничений из-за неизменности $W$, поэтому метод может не так хорошо улавливать зависимости. ConvMixer также выигрывает у Attention по сложности и количеству параметров, однако помимо статичности используемых преобразований одно из них действует более локально (не так, как Attention и ConvMixer), поскольку свёртка видит лишь небольшое пространство. Это компенсируется за счёт увеличения количество слоёв, вместе с которым растёт и receptive field, но всё же Attention в данном плане представляется лучше.

